# 畳み込みと CNN

CNN は、画像を平坦なベクトルとして扱うのではなく、近くの画素どうしの関係を保ったまま特徴を拾います。小さなカーネルを画像の上で滑らせると、線、角、模様のような局所パターンがどこにあるかを反応として残せます。

この教材では、カーネル、padding、stride、pooling を別々の暗記事項としてではなく、画像のどの情報を残し、どの情報を粗くするかという設計として読みます。CNN を使う判断では、局所パターン、位置ずれへの強さ、計算量の釣り合いを同時に見ます。

## 畳み込みは小さな型を滑らせる計算

画像の一部とカーネルの積和を取り、その値を出力位置へ置きます。深層学習ライブラリで Conv2d と呼ばれる演算は、数学的なカーネル反転をせず、相互相関として実装されることが多いです。画像処理として読むなら、どの型に強く反応したかを見る計算です。

In [ ]:
import math
import random
from statistics import mean


def zeros(h, w):
    return [[0.0 for _ in range(w)] for _ in range(h)]


def shape2d(x):
    return (len(x), len(x[0]) if x else 0)


def pad2d(image, pad):
    if pad <= 0:
        return [row[:] for row in image]
    h, w = shape2d(image)
    out = zeros(h + 2 * pad, w + 2 * pad)
    for i in range(h):
        for j in range(w):
            out[i + pad][j + pad] = image[i][j]
    return out


def conv2d(image, kernel, stride=1, padding=0):
    padded = pad2d(image, padding)
    h, w = shape2d(padded)
    kh, kw = shape2d(kernel)
    out_h = (h - kh) // stride + 1
    out_w = (w - kw) // stride + 1
    out = zeros(out_h, out_w)
    for i in range(out_h):
        for j in range(out_w):
            s = 0.0
            for di in range(kh):
                for dj in range(kw):
                    s += padded[i * stride + di][j * stride + dj] * kernel[di][dj]
            out[i][j] = s
    return out


def print_grid(grid, digits=1):
    for row in grid:
        print(' '.join(f'{v:>{digits+4}.{digits}f}' for v in row))

## エッジ検出を手計算に近い形で見る

縦線を含む小さな画像へ、縦方向の変化に反応するカーネルを当てます。出力の符号は、明るさがどちら向きに変化したかを表し、絶対値が大きいほど強い境界を示します。

In [ ]:
image = zeros(8, 8)
for i in range(1, 7):
    image[i][3] = 1.0
    image[i][4] = 1.0
for j in range(1, 7):
    image[5][j] = 1.0

vertical_edge = [
    [-1, 0, 1],
    [-2, 0, 2],
    [-1, 0, 1],
]
horizontal_edge = [
    [-1, -2, -1],
    [0, 0, 0],
    [1, 2, 1],
]

feat_v = conv2d(image, vertical_edge, padding=1)
feat_h = conv2d(image, horizontal_edge, padding=1)
print('input')
print_grid(image, digits=0)
print('vertical response')
print_grid(feat_v, digits=1)
print('horizontal response')
print_grid(feat_h, digits=1)

## padding と stride は出力の形を変える

padding は外周に余白を足して端の情報を扱いやすくします。stride はカーネルを動かす幅です。stride を大きくすると、出力は粗くなり、計算量は下がります。

In [ ]:
def output_shape(h, w, kh, kw, padding=0, stride=1):
    out_h = (h + 2 * padding - kh) // stride + 1
    out_w = (w + 2 * padding - kw) // stride + 1
    return out_h, out_w

for padding, stride in [(0, 1), (1, 1), (1, 2), (2, 2)]:
    out = conv2d(image, vertical_edge, padding=padding, stride=stride)
    print('padding=', padding, 'stride=', stride, 'formula=', output_shape(8, 8, 3, 3, padding, stride), 'actual=', shape2d(out))

## ReLU と pooling は反応を要約する

ReLU は負の反応を 0 にして、強く出た特徴だけを残します。max pooling は小さな範囲の最大値を取り、細かな位置ずれに少し鈍感な表現へ変えます。

In [ ]:
def relu(grid):
    return [[max(0.0, v) for v in row] for row in grid]


def maxpool2d(grid, pool=2, stride=2):
    h, w = shape2d(grid)
    out_h = (h - pool) // stride + 1
    out_w = (w - pool) // stride + 1
    out = zeros(out_h, out_w)
    for i in range(out_h):
        for j in range(out_w):
            vals = []
            for di in range(pool):
                for dj in range(pool):
                    vals.append(grid[i * stride + di][j * stride + dj])
            out[i][j] = max(vals)
    return out

act = relu(feat_v)
pooled = maxpool2d(act)
print('ReLU(vertical response) shape:', shape2d(act))
print_grid(act, digits=1)
print('pooled shape:', shape2d(pooled))
print_grid(pooled, digits=1)

## 畳み込みは重みを使い回す

全結合層は入力位置ごとに別の重みを持ちます。畳み込みは同じカーネルを全位置で使うため、パラメータ数が少なく、画像内の位置が少し変わっても同じ特徴として扱いやすくなります。

In [ ]:
def conv_params(kernel_h, kernel_w, in_channels, out_channels, bias=True):
    return kernel_h * kernel_w * in_channels * out_channels + (out_channels if bias else 0)


def fc_params(input_h, input_w, in_channels, out_units, bias=True):
    return input_h * input_w * in_channels * out_units + (out_units if bias else 0)

fc = fc_params(32, 32, 3, 64)
conv = conv_params(3, 3, 3, 64)
print('fully connected params:', fc)
print('3x3 conv params       :', conv)
print('ratio FC/Conv         :', round(fc / conv, 1))

## 固定フィルタで小さな分類器を作る

横線画像と縦線画像を作り、縦エッジと横エッジの反応を特徴量にします。CNN 全体を学習しなくても、畳み込み特徴が分類に使えることを確認できます。

In [ ]:
def make_stripe(label, size=12, noise=0.08, rng=None):
    rng = rng or random.Random()
    img = zeros(size, size)
    if label == 0:
        row = rng.randrange(2, size - 2)
        for j in range(size):
            img[row][j] = 1.0
            img[row + 1][j] = 1.0
    else:
        col = rng.randrange(2, size - 2)
        for i in range(size):
            img[i][col] = 1.0
            img[i][col + 1] = 1.0
    for i in range(size):
        for j in range(size):
            img[i][j] = min(1.0, max(0.0, img[i][j] + rng.gauss(0.0, noise)))
    return img


def feature_vector(img):
    fv = relu(conv2d(img, vertical_edge, padding=1))
    fh = relu(conv2d(img, horizontal_edge, padding=1))
    max_v = max(max(row) for row in fv)
    max_h = max(max(row) for row in fh)
    avg_v = mean(v for row in fv for v in row)
    avg_h = mean(v for row in fh for v in row)
    return [max_v, max_h, avg_v, avg_h]

rng = random.Random(4)
data = []
for _ in range(300):
    y = rng.randrange(2)
    img = make_stripe(y, rng=rng)
    data.append((feature_vector(img), y))

for k in range(4):
    print('sample', k, 'label=', data[k][1], 'features=', [round(v, 3) for v in data[k][0]])

縦線なら縦方向のエッジ反応が強く、横線なら横方向のエッジ反応が強くなります。線形分類器は、この固定特徴を使って最終判断だけを学習します。

In [ ]:
def sigmoid_scalar(z):
    if z >= 0:
        e = math.exp(-z)
        return 1 / (1 + e)
    e = math.exp(z)
    return e / (1 + e)


def train_logistic(samples, epochs=300, lr=0.08):
    d = len(samples[0][0])
    w = [0.0] * d
    b = 0.0
    for _ in range(epochs):
        gw = [0.0] * d
        gb = 0.0
        for x, y in samples:
            p = sigmoid_scalar(sum(wi * xi for wi, xi in zip(w, x)) + b)
            err = p - y
            for j in range(d):
                gw[j] += err * x[j]
            gb += err
        n = len(samples)
        w = [wi - lr * g / n for wi, g in zip(w, gw)]
        b -= lr * gb / n
    return w, b


def accuracy(samples, w, b):
    correct = 0
    for x, y in samples:
        p = sigmoid_scalar(sum(wi * xi for wi, xi in zip(w, x)) + b)
        correct += int((p >= 0.5) == bool(y))
    return correct / len(samples)

train = data[:240]
test = data[240:]
w, b = train_logistic(train)
print('weights:', [round(v, 3) for v in w], 'bias=', round(b, 3))
print('train acc:', round(accuracy(train, w, b), 3))
print('test acc :', round(accuracy(test, w, b), 3))

## 1x1 畳み込みはチャネルを混ぜる

1x1 conv は隣の画素を見ません。同じ位置のチャネルベクトルに線形変換をかけ、特徴の種類を混ぜ替えます。分類 CNN の特徴マップを、場所ごとのクラススコアへ変換する FCN でも使われます。

In [ ]:
def conv1x1(feature_map, weights, bias):
    # feature_map: H x W x C, weights: O x C
    h = len(feature_map)
    w = len(feature_map[0])
    out_channels = len(weights)
    out = [[[0.0 for _ in range(out_channels)] for _ in range(w)] for _ in range(h)]
    for i in range(h):
        for j in range(w):
            for o in range(out_channels):
                out[i][j][o] = sum(weights[o][c] * feature_map[i][j][c] for c in range(len(weights[o]))) + bias[o]
    return out

feature_map = [
    [[1.0, 0.2], [0.5, 1.4]],
    [[0.1, 1.2], [1.5, 0.3]],
]
weights = [[1.0, -0.5], [-0.2, 0.9], [0.4, 0.4]]
bias = [0.0, 0.1, -0.1]
logits = conv1x1(feature_map, weights, bias)
for row in logits:
    print([[round(v, 3) for v in cell] for cell in row])

畳み込みは小さな模様を全位置で探し、padding と stride は形と細かさを調整し、pooling は反応を要約します。CNN は局所特徴を積み上げることで、少ないパラメータで画像構造を扱えます。1x1 conv は場所を保ったままチャネルを混ぜるため、分類だけでなくセグメンテーションのような場所ごとの予測にもつながります。